# Agentic SOC — setup notebook

This notebook does the same job as `python3 setup_soc.py`, one cell at a time, so you can see what each step does.
Run each cell with **Shift + Enter** and read its output before moving on.

> The step-by-step guide with commands for Windows, macOS and Linux is `docs/SETUP_GUIDE.html` (open it in your browser).
> To open this notebook you need Jupyter: `pip install notebook` then `jupyter notebook docs/SETUP_GUIDE.ipynb`.
> If you'd rather not install Jupyter, just run `python3 setup_soc.py` (Windows: `py setup_soc.py`) in a terminal.

**What you'll end up with:** the FastAPI backend with its LangGraph agents and Milvus RAG store at http://localhost:8000, and the Next.js dashboard at http://localhost:3000.

## 1 · Point the notebook at the project
This finds the project folder (one level above `docs/`) and loads the setup script's functions.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import setup_soc as soc
print("Project folder:", PROJECT)

## 2 · Check your computer
You need **Python 3.10–3.13** (3.12 recommended) and **Node.js 18+**. If Node is missing, the cell tells you how to install it; afterwards restart the notebook kernel (*Kernel → Restart*) so it is found.

In [ ]:
soc.check_prerequisites()

## 3 · Your passwords and keys

Fill in the values below, then run the cell. Leave a key as `""` to skip it.

| Setting | What it is |
|---|---|
| `ANALYST_PASSWORD` | Password for the dashboard sign-in. **Required.** |
| `LLM_PROVIDER` | `"openrouter"` (free hosted Llama 3.3 70B, recommended), `"ollama"` (Llama 3.2 on your CPU), `"groq"`, or `"none"` |
| `OPENROUTER_API_KEY` | Free key from https://openrouter.ai/settings/keys (starts with `sk-or-`) |
| `EMBEDDING_BACKEND` | `"hash"` (no download) or `"fastembed"` (~200 MB, better search) |

Real firewall / Okta / AWS connections are left on the safe **simulator**. Change them later with `python3 setup_soc.py --configure`.

In [ ]:
import getpass, secrets

env = soc.read_env()   # keeps anything you set before
env["ANALYST_USERNAME"] = env.get("ANALYST_USERNAME", "analyst")
env["ANALYST_PASSWORD"] = env.get("ANALYST_PASSWORD") or getpass.getpass("Choose a dashboard password: ")
env["LLM_PROVIDER"] = "openrouter"
env["OPENROUTER_API_KEY"] = env.get("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key (Enter to skip): ")
env.setdefault("OPENROUTER_MODEL", "meta-llama/llama-3.3-70b-instruct:free")
env["EMBEDDING_BACKEND"] = "hash"

env.setdefault("SECRET_KEY", secrets.token_urlsafe(48))
env.setdefault("INGEST_API_KEY", secrets.token_urlsafe(24))
for k, v in {"VECTOR_STORE": "auto", "FIREWALL_PROVIDER": "mock", "IDENTITY_PROVIDER": "mock",
             "AUTO_APPROVE": "false", "DEMO_MODE": "true",
             "CORS_ORIGINS": "http://localhost:3000,http://127.0.0.1:3000"}.items():
    env.setdefault(k, v)

soc.write_env(env)
print("Saved to", soc.ENV_FILE, "- keys stored:", sorted(k for k, v in env.items() if v and "KEY" in k))

## 4 · Install the backend
Creates a private Python environment in `backend/.venv` and installs FastAPI, LangGraph, pymilvus/Milvus Lite and boto3. Takes a few minutes the first time.

In [ ]:
soc.install_backend(env)

## 5 · Install the dashboard
Runs `npm install` in `frontend/`.

In [ ]:
soc.install_frontend()

## 6 · Self-test
Runs the backend tests: intel import, guardrails, RAG search, the full agent pipeline, approvals and the HTTP API. All should pass.

In [ ]:
soc.run_tests()

## 7 · Start it

This starts both servers and opens http://localhost:3000. Sign in with your username and password, then press **Simulate alert**.

The cell keeps running while the platform is up. To stop, use **Kernel → Interrupt** (the ■ button).

In [ ]:
soc.start(open_browser=True)

## 8 · Optional next steps

- **Publish to GitHub + Docker Hub:** in a terminal run `python3 setup_soc.py --github`. It needs a GitHub token (repo + workflow scopes) and a Docker Hub access token, and only ever asks for those.
- **Run with Docker:** `python3 setup_soc.py --docker`
- **Kubernetes on your laptop:** see `deploy/terraform/README.md`
- **Something wrong?** `python3 setup_soc.py --doctor`